# Tutorial -  Volatilidad
Sergio Cabrales, Universidad de los Andes

https://www.sac2.com/

## 1. Carga de librerías, funciones y APIs necesarias.

#### 1.1. Instalan las librerías que no incluye Google Colab

In [ ]:
pip install yfinance

In [ ]:
pip install mplfinance

#### 1.2. Se cargan las librerías requeridas

In [ ]:
# Funciones numéricas adicionales
import numpy as np

# Lectura de datos y manejo de Data-sets
import pandas as pd

# Datos
import yfinance as yfin

# Gráficos
import matplotlib.pyplot as plt

#analisis tecnico
import mplfinance as mpf

# Probabilidad y estadística
import math
from scipy.stats import norm

## 2. Obtención de datos históricos

#### 2.1. Descarga de datos desde Yahoo Finance

https://finance.yahoo.com/


In [ ]:
# Descargamos datos de la acción sleccionada:
df = yfin.download('COP=X', start='2020-01-01', multi_level_index=False)
df

In [ ]:
# Plotting the 'Close' column
plt.figure(figsize=(15, 8))  # Adjust figure size as needed
ax = plt.gca() # Get current axes
plt.plot(df['Close'], color='blue')
plt.title('USD/COP')
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.grid(True)  # Optional: Add a grid for better readability

# Remove top and right spines for a 'The Economist' like style
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()

In [ ]:
df.to_excel('USD_COP.xlsx')

## 4. Retornos

### 4.1. Retornos Logarítmicos

Los retornos logarítmicos se calculan como:
$$
r_{t} = ln \left( \frac{S_t}{S_{t-1}} \right ) = ln \left( S_{t} \right) - ln \left( S_{t-1} \right)
$$

In [ ]:
# Guardamos los retornos logaritmicos en una nueva columna.
df['Log Returns'] = np.log(df['Close']) - np.log(df['Close'].shift(1))
df['Log Returns'][0] = 0
df

### 4.3. Retornos Logarítmicos anualizados

Podemos calcular el log-retorno anual ($r$) como el número de días bursátiles (252 días) por el promedio del log-retorno diario:

$$
r = 252 \bar{r_t}
$$

In [ ]:
# Podemos imprimir el retornos anual:
LogReturns = np.mean(df["Log Returns"])*252
LogReturns

### 4.4. Gráfica de retornos
- Podemos graficar los retornos igual que como graficamos los precios.

In [ ]:
# Gráfico de los retornos logarítmicos
plt.figure(figsize=(15,8))
plt.plot(df['Log Returns'], color = 'red')
plt.title('Retornos Logarítmicos')
plt.xlabel('Fecha')
plt.show()

## 5. Volatilidad

### 5.1 Volatilidad diaria y anual

La volatilidad diaria del activo es la desviación estándar de sus retornos o la raíz de la varianza:  

$$vol=desv(r)=\sqrt{Var(r)}$$

En finanzas, se utiliza con mayor frecuencia la volatilidad anualizada ($\sigma$) en lugar de la volatilidad diaria. Teniendo en cuenta que en cada año hay 252 días bursátiles:

$$ \sigma^{2} = \sum_{1}^{252} Var_{diaria}$$
$$ \sigma^{2} = 252 \sigma_{diaria}^{2}$$

Se saca la raíz cuadra a ambos lados para calcular la volatilidad:

$$ \sqrt{\sigma^{2}} = \sqrt{252 \sigma_{diaria}^{2}}$$
$$ \sigma = \sigma_{diaria} \sqrt{252}$$

In [ ]:
# Calculamos la volatilidad diaria con los retornos logaritmicos.
vol_d = np.std(df['Log Returns'])

# Anualizamos la volatilidad diaria.
vol_a = vol_d * np.sqrt(252)

print("Volatilidad diaria: {:.4f} %".format(100*vol_d))
print("Volatilidad anualizada: {:.4f} %".format(100*vol_a))

## 6. Black-Scholes-Merton model

### 6.1. Función de Black-Scholes model

In [ ]:
def black_scholes_i(S, K, r, rf, T, sigma, option):
    """
    Calculate the price of a European call or put option using the Black-Scholes model.
    Parameters:
        S (float): underlying asset price
        K (float): option strike price
        r (float): risk-free interest rate
        rf (float): risk-free interest rate - foreign
        T (float): time to maturity in years
        sigma (float): volatility of underlying asset returns
        option (str): type of option to be priced, either 'call' (default) or 'put'
    Returns:
        price (float): price of the option
    """
    d1 = (np.log(S) - np.log(K) + (r - rf + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option == 'call':
        price = np.exp(-r * T) * (S * np.exp((r-rf) * T)*norm.cdf(d1) - K * norm.cdf(d2))
    elif option == 'put':
        price = np.exp(-r * T) * (K * norm.cdf(-d2) - S * np.exp((r-rf) * T)*norm.cdf(-d1))
    else:
        raise ValueError("Invalid option type. Choose 'call' or 'put")

    return price

### 6.2. Ejemplos

In [ ]:
# Example usage
S = df['Close'].iloc[-1]    # current stock price
K = 4000    # strike price
r = 0.1025    # risk-free interest rate
rf = 0.0365    # risk-free interest rate - foreign
T = 10/12     # time to maturity in years
sigma = vol_a # volatility

call_price = black_scholes_i(S, K, r, rf, T, sigma, 'call')
put_price = black_scholes_i(S, K, r, rf, T, sigma, 'put')

print("Call option price:", call_price)
print("Put option price:", put_price)

## 7. Futures Prices

In [ ]:
F_0= S*np.exp((r-rf)*T)
F_0

In [ ]:
from dateutil.relativedelta import relativedelta
import pandas as pd
last_historical_date = df.index[-1]
# Get the first day of the month after the last historical date
first_future_month = (last_historical_date + relativedelta(months=1)).replace(day=1)

# Generate future dates until December 2026
end_date = pd.Timestamp('2027-01-01')
future_dates = pd.date_range(start=first_future_month, end=end_date, freq='MS')
# Calculate 'T' (time in years) for each future date
T_values = [(date - last_historical_date).days / 365.0 for date in future_dates]

future_prices = [S * np.exp((r - rf) * t_val) for t_val in T_values]

In [ ]:
plt.figure(figsize=(15, 8))

# Plot historical 'Close' prices
plt.plot(df.index, df['Close'], color='blue', label='Historical Close Price')

# Plot future prices
plt.plot(future_dates, future_prices, color='red', linestyle='--', label='Future Price')

plt.title('USD/COP Historical and Futures Prices')
plt.xlabel('Date')
plt.ylabel('Price')
plt.grid(True)
plt.legend()

# Remove top and right spines for a 'The Economist' like style
ax = plt.gca() # Get current axes
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()

## Summary:

### Data Analysis Key Findings

*   The last historical date in the dataset was identified as `2026-02-23`.
*   A total of 10 future monthly dates were generated, starting from `2026-03-01` (the first day of the month after the last historical date) and extending until December 2026.
*   For each future date, the time in years ('T') from the last historical date was calculated, with the first future date having a 'T' value of approximately `0.0164` years.
*   Corresponding future prices were calculated using the formula \(S \cdot e^{(r-rf)T}\), based on the generated 'T' values.
*   A visualization was created displaying both the historical 'Close' prices and the projected future prices, showing the trend continuation.

### Insights or Next Steps

*   The generated projections provide a forward-looking view of the USD/COP prices based on the given formula and parameters, which can be valuable for planning and decision-making.
*   Further analysis could involve performing sensitivity tests on the parameters \(S\), \(r\), and \(rf\) to understand how different assumptions impact the future price projections.
